# RCA Project: Implementation and Analysis of Linear Regression Using Python

**Subject:** Regression & Classification Analysis (RCA)  
**Project Topic:** Linear Regression  
**Dataset:** California Housing Dataset  
**Platform:** Google Colab / Jupyter Notebook  
**Language:** Python

## Aim
To implement Simple and Multiple Linear Regression on a real-world dataset, visualize relationships between features and house values, predict continuous values, and evaluate model performance using MAE, MSE, RMSE, and R² score.

## Objectives
- Load and understand a regression dataset.
- Perform basic data checking and exploratory data analysis (EDA).
- Implement Simple Linear Regression using one feature.
- Implement Multiple Linear Regression using several features.
- Evaluate and compare both models.
- Predict house value for a new sample.
- Export the dataset and predictions as CSV files.

**Note:** The target `MedHouseVal` is measured in units of $100,000. For example, a value of 2.0 represents approximately $200,000. This dataset's target has a known cap, so predictions at the upper end should be interpreted cautiously.

## 1. Import Required Libraries
We use Pandas and NumPy for data handling, Matplotlib and Seaborn for visualization, and Scikit-learn for loading the dataset, training, and evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries imported successfully!")

## 2. Load the Dataset
The California Housing dataset contains housing-related numerical features and a continuous target variable.

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()

print("Dataset loaded successfully!")

## 3. Display First Five Records

In [ ]:
df.head()

## 4. Dataset Shape and Column Names
`shape` shows the number of rows and columns. The target is `MedHouseVal`; the remaining columns are input features.

In [ ]:
print("Dataset shape:", df.shape)
print("\\nColumn names:")
print(df.columns.tolist())
print("\\nTarget column: MedHouseVal")

## 5. Dataset Information
This shows each column's data type and the count of non-missing values.

In [ ]:
df.info()

## 6. Statistical Summary
The `describe()` method gives count, mean, standard deviation, minimum, quartiles and maximum.

In [ ]:
df.describe()

## 7. Missing Values and Duplicate Records
We check whether data cleaning is needed. This dataset normally has no missing values and no exact duplicate rows.

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

print("\\nNumber of duplicate rows:", df.duplicated().sum())

## 8. Exploratory Data Analysis (EDA)
We inspect distributions of important columns using histograms.

In [ ]:
df.hist(figsize=(14, 10), bins=30, edgecolor="black")
plt.suptitle("Distribution of California Housing Features", fontsize=16)
plt.tight_layout()
plt.show()

## 9. Relationship Between Median Income and House Value
Median income is used as the single input feature for Simple Linear Regression.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["MedInc"], df["MedHouseVal"], alpha=0.25)
plt.xlabel("Median Income")
plt.ylabel("Median House Value (in $100,000 units)")
plt.title("Median Income vs Median House Value")
plt.grid(alpha=0.2)
plt.show()

## 10. Correlation Heatmap
Correlation shows the strength and direction of linear association between numerical columns. Correlation alone does not prove causation.

In [ ]:
plt.figure(figsize=(11, 8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 11. Simple Linear Regression
We use only `MedInc` to predict `MedHouseVal`. The dataset is split into training data (80%) and testing data (20%). `random_state=42` makes the split reproducible.

In [ ]:
X_simple = df[["MedInc"]]
y = df["MedHouseVal"]

X_train_s, X_test_s, y_train, y_test = train_test_split(
    X_simple, y, test_size=0.20, random_state=42
)

simple_model = LinearRegression()
simple_model.fit(X_train_s, y_train)

y_pred_simple = simple_model.predict(X_test_s)

print("Slope (coefficient):", simple_model.coef_[0])
print("Intercept:", simple_model.intercept_)
print("Simple model trained successfully!")

### 11.1 Plot the fitted regression line
The line represents the model's estimated relationship between median income and house value.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(X_test_s["MedInc"], y_test, alpha=0.3, label="Actual test data")

order = np.argsort(X_test_s["MedInc"].values)
x_sorted = X_test_s["MedInc"].values[order]
y_sorted = y_pred_simple[order]
plt.plot(x_sorted, y_sorted, color="red", linewidth=2, label="Regression line")

plt.xlabel("Median Income")
plt.ylabel("Median House Value (in $100,000 units)")
plt.title("Simple Linear Regression: Income vs House Value")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 12. Multiple Linear Regression
We use all eight input features to predict `MedHouseVal`.

In [ ]:
X_multiple = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multiple, y, test_size=0.20, random_state=42
)

multiple_model = LinearRegression()
multiple_model.fit(X_train_m, y_train_m)

y_pred_multiple = multiple_model.predict(X_test_m)

print("Multiple Linear Regression model trained successfully!")
print("Intercept:", multiple_model.intercept_)

### 12.1 Model Coefficients
Each coefficient estimates the change in predicted target for a one-unit change in that feature, holding the other features fixed (within the model).

In [ ]:
coefficients = pd.DataFrame({
    "Feature": X_multiple.columns,
    "Coefficient": multiple_model.coef_
}).sort_values(by="Coefficient", ascending=False)

coefficients

## 13. Evaluation Metrics
- **MAE:** Average absolute prediction error.
- **MSE:** Average squared error; larger errors are penalized more.
- **RMSE:** Square root of MSE, expressed in target units.
- **R²:** Fraction of test-target variation explained by the model relative to a mean baseline (can be negative on test data).

In [ ]:
def evaluate_model(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    return {
        "Model": model_name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2 Score": r2
    }

simple_metrics = evaluate_model(y_test, y_pred_simple, "Simple Linear Regression")
multiple_metrics = evaluate_model(y_test_m, y_pred_multiple, "Multiple Linear Regression")

results = pd.DataFrame([simple_metrics, multiple_metrics])
results

## 14. Compare the Models
A lower MAE, MSE, and RMSE indicates smaller prediction errors. A higher R² indicates better fit relative to the baseline on this test set. Compare all metrics rather than relying only on R².

In [ ]:
results.set_index("Model")[["MAE", "RMSE", "R2 Score"]].plot(
    kind="bar", figsize=(9, 5), rot=0
)
plt.title("Model Performance Comparison")
plt.ylabel("Metric Value")
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

## 15. Actual vs Predicted Values (Multiple Regression)
Points closer to the diagonal line indicate predictions closer to actual values.

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test_m, y_pred_multiple, alpha=0.25)

low = min(y_test_m.min(), y_pred_multiple.min())
high = max(y_test_m.max(), y_pred_multiple.max())
plt.plot([low, high], [low, high], "r--", label="Ideal prediction")

plt.xlabel("Actual House Value (in $100,000 units)")
plt.ylabel("Predicted House Value (in $100,000 units)")
plt.title("Actual vs Predicted: Multiple Linear Regression")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 16. Residual Plot
A residual is actual value minus predicted value. A pattern in residuals may indicate that the linear model is missing structure.

In [ ]:
residuals = y_test_m - y_pred_multiple

plt.figure(figsize=(8, 5))
plt.scatter(y_pred_multiple, residuals, alpha=0.25)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted House Value")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residual Plot")
plt.grid(alpha=0.2)
plt.show()

## 17. Predict for a New Example
Enter a sample row using the same feature order and units. This is an illustrative example, not a real listing valuation.

In [ ]:
new_house = pd.DataFrame([{
    "MedInc": 5.0,
    "HouseAge": 25.0,
    "AveRooms": 5.0,
    "AveBedrms": 1.0,
    "Population": 1000.0,
    "AveOccup": 3.0,
    "Latitude": 34.0,
    "Longitude": -118.0
}])

predicted_value = multiple_model.predict(new_house)[0]

print(f"Predicted median house value: {predicted_value:.3f} ($100,000 units)")
print(f"Approximate value in USD: ${predicted_value * 100000:,.0f}")

## 18. Export Dataset, Predictions and Metrics to CSV
The following files are created when the notebook runs.

In [ ]:
df.to_csv("california_housing_dataset.csv", index=False)

prediction_output = pd.DataFrame({
    "Actual": y_test_m.reset_index(drop=True),
    "Predicted": pd.Series(y_pred_multiple)
})
prediction_output.to_csv("actual_vs_predicted.csv", index=False)

results.to_csv("model_evaluation_results.csv", index=False)

print("CSV files exported:")
print("- california_housing_dataset.csv")
print("- actual_vs_predicted.csv")
print("- model_evaluation_results.csv")

## 19. Download Files from Google Colab
Run this cell to download the dataset CSV. You may download the other outputs by changing the filename.

In [ ]:
from google.colab import files
files.download("california_housing_dataset.csv")

## 20. Conclusion

In this project, Simple Linear Regression and Multiple Linear Regression were implemented using the California Housing dataset. Exploratory data analysis and visualizations helped examine the relationships between housing features and the target value. The models were evaluated using MAE, MSE, RMSE, and R². Multiple Linear Regression uses several features, so it can capture more information than a one-feature model; however, its performance should be judged from the test-set metrics. This project demonstrates the use of Linear Regression for predicting a continuous numerical value.

## Limitations
- Linear Regression assumes a linear relationship between features and target.
- This dataset represents aggregated census-block information, not individual houses.
- The target values are capped in the dataset.
- The predictions are educational estimates and should not be used as real property valuations.

## Submission Checklist
- [ ] Run all notebook cells in Google Colab.
- [ ] Check that outputs and plots appear.
- [ ] Download the notebook (`.ipynb`).
- [ ] Download generated CSV files.
- [ ] Save the README and requirements files.
- [ ] Add project files to GitHub if required by faculty.